### **Day 10: Complex Data Types (Arrays, Structs, and Maps)**

Yesterday, we mastered Spark SQL and saw how to query our distributed data using standard relational strings. Today, we wrap up Phase 3 of our curriculum by tackling a massive real-world data engineering challenge: **Complex Data Types**.

In production environments, data is rarely flat or clean. You will constantly deal with nested log files, semi-structured API payloads, and JSON arrays. To be an expert, you must know how to manipulate multi-layered structures—such as Arrays, Maps, and Structs—without destroying the scalability of your cluster.

**Today's Objective**

By the end of this session, you will understand the structural difference between Arrays, Structs, and Maps in PySpark, how to flatten nested JSON architectures efficiently, and how to execute schema-aware row explosions.

**1. Deconstructing the Three Complex Types**

PySpark breaks complex, semi-structured data down into three primary data types. Each handles nested information differently:

*A. The ArrayType (The List)*

An ArrayType is a collection of elements that all share the exact same data type. Think of it as a standard Python list embedded inside a single table cell.

* *Example:* A column named `tags` might contain `["electronics", "sale", "wireless"]` for a specific product row.

*B. The StructType (The Nested Object)*

A StructType represents a nested structure or an "object" within a row. It allows you to group related columns together inside a single parent column, creating a mini-table inside a cell. Each field inside a Struct can have a completely different data type.

* *Example:* A column named `address` could be a Struct containing sub-columns like `city` (String), `zip_code` (Integer), and `state` (String).

*C. The MapType (The Key-Value Dictionary)*

A MapType is a collection of key-value pairs, completely analogous to a standard Python dictionary. All keys must share the same data type (usually strings), and all values must share the same data type.

* *Example:* A column named `attributes` might store variable properties like `{"color": "red", "size": "XL", "material": "cotton"}`.



**2. Flattening and Exploding Complex Structures**

When processing big data, your ultimate goal is usually to transform nested semi-structured data into flat, relational tables so analysts can run standard SQL queries against them. PySpark provides unique structural transformations to handle this.

*Unpacking Structs (The Dot Notation)*

Unpacking a nested Struct is computationally very cheap because it does not change the row count of your dataset. You can access nested child fields instantly using standard dot notation (`parent_column.child_column`). Spark simply splits the internal sub-columns out into separate, flat columns on the fly.

*Exploding Arrays and Maps (The Row-Multiplication Shift)*

Handling Arrays and Maps is a completely different architectural story. If a single customer row contains an array of 5 purchased items, you cannot easily query those items side-by-side. To fix this, PySpark uses an operation called `explode()`.

When you explode a column containing an array, PySpark duplicates the original row for *every single element* inside that array.

* If a row has an array with 3 items, `explode()` destroys that single row and creates 3 independent rows.
* Each new row copies all the original parent data (like `customer_id` or `date`) but contains a single, isolated element from the array.

> **Expert Architectural Warning:** Using `explode()` on a massive dataset can exponentially increase your total row count in an instant. If you explode large arrays across millions of rows, you will dramatically increase the size of your dataset in memory, which can lead to severe memory saturation and execution slowdowns across your Executors.

**3. High-Performance JSON Ingestion**

In production pipelines, you will often read raw text logs where an entire column is just a giant stringified JSON text block.

Instead of writing slow, custom Python loops to parse that text, PySpark provides highly optimized built-in functions:

* **`from_json()`**: This function converts a text column containing a JSON string into a fully structured Spark Struct or Map. To use it, you must pass an explicit schema blueprint so Spark knows exactly how to map the JSON keys to typed columns.
* **`to_json()`**: The inverse operation. It takes a highly complex, nested structure and compresses it down into a single flat stringified JSON text block, making it clean and easy to stream out to external APIs or message queues.